# 🏷️ Gene Ontology (GO) Analysis

---

## Learning Objectives

- Understand GO structure and terminology
- Navigate the three GO domains
- Perform GO enrichment analysis
- Interpret GO results

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook is designed to bridge concept to practical analysis workflows.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Inspect assumptions before running model/statistical code.
- Track input/output files for reproducibility.
- Interpret outputs with biological context, not numbers alone.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

## Gene Ontology Structure

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    GENE ONTOLOGY (GO)                                   │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   GO = Controlled vocabulary for describing gene/protein function       │
│                                                                         │
│   THREE DOMAINS (ONTOLOGIES):                                           │
│                                                                         │
│   ┌─────────────────────────────────────────────────────────────────┐   │
│   │ 1. MOLECULAR FUNCTION (MF)                                      │   │
│   │    What does the gene product DO?                               │   │
│   │    Examples: kinase activity, DNA binding, transporter          │   │
│   │                                                                 │   │
│   │ 2. BIOLOGICAL PROCESS (BP)                                      │   │
│   │    What pathway/process is it part of?                          │   │
│   │    Examples: cell cycle, apoptosis, signal transduction         │   │
│   │                                                                 │   │
│   │ 3. CELLULAR COMPONENT (CC)                                      │   │
│   │    WHERE does it localize?                                      │   │
│   │    Examples: nucleus, mitochondrion, plasma membrane            │   │
│   └─────────────────────────────────────────────────────────────────┘   │
│                                                                         │
│   GO terms form a DIRECTED ACYCLIC GRAPH (DAG):                         │
│                                                                         │
│   biological_process (GO:0008150)                                       │
│        │                                                                │
│        └── cellular process (GO:0009987)                                │
│             │                                                           │
│             └── cell cycle (GO:0007049)                                 │
│                  │                                                      │
│                  ├── mitotic cell cycle (GO:0000278)                    │
│                  │    │                                                 │
│                  │    └── G1/S transition (GO:0000082)                  │
│                  │                                                      │
│                  └── meiotic cell cycle (GO:0051321)                    │
│                                                                         │
│   Relationship: is_a, part_of, regulates                                │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Example GO term data
GO_TERMS = {
    # Molecular Function
    'GO:0003674': {'name': 'molecular_function', 'domain': 'MF', 'parents': []},
    'GO:0003824': {'name': 'catalytic activity', 'domain': 'MF', 'parents': ['GO:0003674']},
    'GO:0016301': {'name': 'kinase activity', 'domain': 'MF', 'parents': ['GO:0016772']},
    'GO:0016772': {'name': 'transferase activity', 'domain': 'MF', 'parents': ['GO:0003824']},
    'GO:0004672': {'name': 'protein kinase activity', 'domain': 'MF', 'parents': ['GO:0016301']},
    
    # Biological Process
    'GO:0008150': {'name': 'biological_process', 'domain': 'BP', 'parents': []},
    'GO:0009987': {'name': 'cellular process', 'domain': 'BP', 'parents': ['GO:0008150']},
    'GO:0007049': {'name': 'cell cycle', 'domain': 'BP', 'parents': ['GO:0009987']},
    'GO:0000278': {'name': 'mitotic cell cycle', 'domain': 'BP', 'parents': ['GO:0007049']},
    'GO:0006915': {'name': 'apoptotic process', 'domain': 'BP', 'parents': ['GO:0009987']},
    
    # Cellular Component
    'GO:0005575': {'name': 'cellular_component', 'domain': 'CC', 'parents': []},
    'GO:0005622': {'name': 'intracellular', 'domain': 'CC', 'parents': ['GO:0005575']},
    'GO:0005634': {'name': 'nucleus', 'domain': 'CC', 'parents': ['GO:0005622']},
    'GO:0005737': {'name': 'cytoplasm', 'domain': 'CC', 'parents': ['GO:0005622']},
    'GO:0005739': {'name': 'mitochondrion', 'domain': 'CC', 'parents': ['GO:0005737']}
}

# Gene annotations (simplified)
GENE_GO_ANNOTATIONS = {
    'TP53': ['GO:0006915', 'GO:0007049', 'GO:0005634'],  # Apoptosis, cell cycle, nucleus
    'CDK2': ['GO:0004672', 'GO:0000278', 'GO:0005634'],  # Kinase, mitotic, nucleus
    'BCL2': ['GO:0006915', 'GO:0005739'],               # Apoptosis, mitochondrion
    'CASP3': ['GO:0006915', 'GO:0003824', 'GO:0005737'], # Apoptosis, catalytic, cytoplasm
    'CCNB1': ['GO:0000278', 'GO:0005634', 'GO:0005737']  # Mitotic, nucleus, cytoplasm
}

def get_go_term_info(go_id):
    """Get information about a GO term."""
    term = GO_TERMS.get(go_id)
    if term:
        return f"{go_id}: {term['name']} [{term['domain']}]"
    return f"{go_id}: Unknown"

def get_gene_annotations(gene):
    """Get GO annotations for a gene."""
    return GENE_GO_ANNOTATIONS.get(gene.upper(), [])

# Example
for gene in ['TP53', 'CDK2', 'BCL2']:
    print(f"\n{gene} annotations:")
    for go_id in get_gene_annotations(gene):
        print(f"  {get_go_term_info(go_id)}")

---

## GO Evidence Codes

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    GO EVIDENCE CODES                                    │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Experimental Evidence:                                                │
│   ┌────────┬──────────────────────────────────────────────────┐         │
│   │ EXP    │ Inferred from Experiment                         │         │
│   │ IDA    │ Inferred from Direct Assay                       │         │
│   │ IPI    │ Inferred from Physical Interaction               │         │
│   │ IMP    │ Inferred from Mutant Phenotype                   │         │
│   │ IGI    │ Inferred from Genetic Interaction                │         │
│   │ IEP    │ Inferred from Expression Pattern                 │         │
│   └────────┴──────────────────────────────────────────────────┘         │
│                                                                         │
│   Computational Evidence:                                               │
│   ┌────────┬──────────────────────────────────────────────────┐         │
│   │ ISS    │ Inferred from Sequence Similarity                │         │
│   │ ISO    │ Inferred from Sequence Orthology                 │         │
│   │ ISA    │ Inferred from Sequence Alignment                 │         │
│   │ ISM    │ Inferred from Sequence Model                     │         │
│   │ IGC    │ Inferred from Genomic Context                    │         │
│   │ IBA    │ Inferred from Biological Aspect of Ancestor      │         │
│   └────────┴──────────────────────────────────────────────────┘         │
│                                                                         │
│   Curator/Author Evidence:                                              │
│   ┌────────┬──────────────────────────────────────────────────┐         │
│   │ TAS    │ Traceable Author Statement                       │         │
│   │ NAS    │ Non-traceable Author Statement                   │         │
│   └────────┴──────────────────────────────────────────────────┘         │
│                                                                         │
│   Automatic:                                                            │
│   ┌────────┬──────────────────────────────────────────────────┐         │
│   │ IEA    │ Inferred from Electronic Annotation              │         │
│   └────────┴──────────────────────────────────────────────────┘         │
│                                                                         │
│   Quality hierarchy: Experimental > Computational > Electronic          │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
EVIDENCE_CODES = {
    # Experimental
    'EXP': ('Experiment', 'experimental', 5),
    'IDA': ('Direct Assay', 'experimental', 5),
    'IPI': ('Physical Interaction', 'experimental', 5),
    'IMP': ('Mutant Phenotype', 'experimental', 5),
    'IGI': ('Genetic Interaction', 'experimental', 5),
    'IEP': ('Expression Pattern', 'experimental', 4),
    
    # Computational
    'ISS': ('Sequence Similarity', 'computational', 3),
    'ISO': ('Sequence Orthology', 'computational', 3),
    'ISA': ('Sequence Alignment', 'computational', 3),
    'ISM': ('Sequence Model', 'computational', 3),
    'IBA': ('Biological Ancestor', 'computational', 3),
    
    # Curator
    'TAS': ('Author Statement (Traceable)', 'curator', 2),
    'NAS': ('Author Statement (Non-traceable)', 'curator', 1),
    
    # Electronic
    'IEA': ('Electronic Annotation', 'automatic', 1)
}

def evidence_quality(code):
    """Get evidence quality score (1-5)."""
    info = EVIDENCE_CODES.get(code)
    return info[2] if info else 0

def filter_by_evidence(annotations, min_quality=3):
    """
    Filter GO annotations by evidence quality.
    
    Parameters:
        annotations: List of (go_id, evidence_code) tuples
        min_quality: Minimum evidence quality (1-5)
    """
    return [(go_id, ec) for go_id, ec in annotations 
            if evidence_quality(ec) >= min_quality]

# Example
annotations = [
    ('GO:0006915', 'IDA'),   # Direct assay
    ('GO:0007049', 'IEA'),   # Electronic
    ('GO:0005634', 'IMP'),   # Mutant phenotype
]

print("All annotations:")
for go_id, ec in annotations:
    info = EVIDENCE_CODES.get(ec, ('Unknown', 'unknown', 0))
    print(f"  {go_id} - {ec} ({info[0]}, quality: {info[2]})")

print("\nFiltered (quality >= 3):")
filtered = filter_by_evidence(annotations, min_quality=3)
for go_id, ec in filtered:
    print(f"  {go_id} - {ec}")

---

## GO Enrichment Analysis

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    GO ENRICHMENT                                        │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Similar to pathway enrichment, but for GO terms                       │
│                                                                         │
│   Additional considerations:                                            │
│                                                                         │
│   1. PROPAGATION: Genes annotated to a term are also annotated          │
│      to all parent terms (true path rule)                               │
│                                                                         │
│      Gene → cell cycle (GO:0007049)                                     │
│          └→ cellular process (GO:0009987)  ← propagated                 │
│              └→ biological_process (GO:0008150)  ← propagated           │
│                                                                         │
│   2. REDUNDANCY: Child terms may overlap with parent terms              │
│      Solution: Use semantic similarity to group similar terms           │
│                                                                         │
│   3. TERM SPECIFICITY: More specific terms are often more               │
│      informative but have fewer genes                                   │
│      Solution: Balance between specificity and statistical power        │
│                                                                         │
│   Common tools:                                                         │
│   • DAVID                                                               │
│   • g:Profiler                                                          │
│   • clusterProfiler (R)                                                 │
│   • GOATOOLS (Python)                                                   │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
from scipy import stats

def propagate_annotations(gene_annotations, go_terms):
    """
    Propagate annotations to parent terms.
    """
    propagated = {}
    
    for gene, go_ids in gene_annotations.items():
        all_terms = set(go_ids)
        
        # Propagate each term to parents
        to_process = list(go_ids)
        while to_process:
            term = to_process.pop()
            if term in go_terms:
                for parent in go_terms[term]['parents']:
                    if parent not in all_terms:
                        all_terms.add(parent)
                        to_process.append(parent)
        
        propagated[gene] = list(all_terms)
    
    return propagated

def go_enrichment(gene_list, gene_annotations, go_terms, background_size=20000):
    """
    Perform GO enrichment analysis.
    """
    # Propagate annotations
    propagated = propagate_annotations(gene_annotations, go_terms)
    
    # Build term→genes mapping
    term_to_genes = {}
    for gene, terms in propagated.items():
        for term in terms:
            if term not in term_to_genes:
                term_to_genes[term] = set()
            term_to_genes[term].add(gene)
    
    gene_set = set(g.upper() for g in gene_list)
    n = len(gene_set)
    N = background_size
    
    results = []
    
    for term_id, term_genes in term_to_genes.items():
        K = len(term_genes) * 100  # Scale for example
        overlap = gene_set & set(g.upper() for g in term_genes)
        k = len(overlap)
        
        if k == 0:
            continue
        
        p_value = stats.hypergeom.sf(k - 1, N, K, n)
        
        term_info = go_terms.get(term_id, {})
        results.append({
            'go_id': term_id,
            'name': term_info.get('name', 'Unknown'),
            'domain': term_info.get('domain', '?'),
            'p_value': p_value,
            'overlap': k,
            'genes': list(overlap)
        })
    
    results.sort(key=lambda x: x['p_value'])
    return results[:10]  # Top 10

# Example
test_genes = ['TP53', 'CDK2', 'BCL2', 'CASP3', 'CCNB1']
results = go_enrichment(test_genes, GENE_GO_ANNOTATIONS, GO_TERMS)

print("GO Enrichment Results:")
print("=" * 60)
for r in results[:5]:
    print(f"\n{r['go_id']}: {r['name']} [{r['domain']}]")
    print(f"  P-value: {r['p_value']:.2e}")
    print(f"  Genes: {', '.join(r['genes'])}")

---

## 🏋️ Exercises

### Exercise 1: GO Slim
Create a GO slim (simplified ontology) for your data.

### Exercise 2: Semantic Similarity
Calculate similarity between GO terms.

### Exercise 3: Visualization
Create a GO graph visualization.

---

## 📚 Resources

- [Gene Ontology](http://geneontology.org/) - Main resource
- [AmiGO](http://amigo.geneontology.org/) - GO browser
- [g:Profiler](https://biit.cs.ut.ee/gprofiler/) - Enrichment analysis
- [QuickGO](https://www.ebi.ac.uk/QuickGO/) - GO browser